# Probabilistic Bayesian Neural Networks

**Author:** [Jose Robledo](https://jorobledo.github.io/)<br>
**Date created:** 20.01.2026<br>
**Last modified:** 20.01.2025<br>
**Description:** Building probabilistic Bayesian neural network models with Bayesian-torch.

## Introduction

This notebook is a Bayesian-torch adaptation of the [notebook from Khalid Salama](https://github.com/ksalama/keras-io/blob/2549b0afb720f9b6f7e3b6c82dcb456472101539/examples/keras_recipes/ipynb/bayesian_neural_networks.ipynb#L9).

Taking a probabilistic approach to deep learning allows to account for *uncertainty*,
so that models can assign less levels of confidence to incorrect predictions.
Sources of uncertainty can be found in the data, due to measurement error or
noise in the labels, or the model, due to insufficient data availability for
the model to learn effectively.


This example demonstrates how to build basic probabilistic Bayesian neural networks
to account for these two types of uncertainty.
We will use [Bayesian-Torch](https://github.com/IntelLabs/bayesian-torch) library.

You can install `Bayesian-Torch` using the following command:

```bash
pip install bayesian-torch
```

## The dataset

We use the [Wine Quality](https://archive.ics.uci.edu/ml/datasets/wine+quality)
dataset.


We use the white wine subset, which contains 4,898 examples.
The dataset has 11 numerical physicochemical features of the wine, and the task
is to predict the wine quality, which is a score between 0 and 10.
In this example, we treat this as a regression task.

## Setup

In [1]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.optim import RMSprop
import torch.nn.functional as F
from torch.distributions import Normal, Independent
from torch.utils.data import TensorDataset, DataLoader

from bayesian_torch.layers import LinearReparameterization

## Create training and evaluation datasets

Here, we load the `wine_quality` dataset using `tfds.load()`, and we convert
the target feature to float. Then, we shuffle the dataset and split it into
training and test sets. We take the first `train_size` examples as the train
split, and the rest as the test split.

In [2]:
def get_train_and_test_dataloaders(
    train_size: float,
    batch_size: int,
    seed: int = 0,
    num_workers: int = 0,
    pin_memory: bool = True,
):
    """
    train_size: float in (0, 1], proportion of data used for training (e.g. 0.8)
    batch_size: batch size for both DataLoaders
    seed: RNG seed for reproducible shuffling (train split only)
    """

    if not (0.0 < train_size <= 1.0):
        raise ValueError(f"train_size must be in (0, 1], got {train_size}")

    url_red = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"
    df = pd.read_csv(url_red, sep=";")

    if "quality" not in df.columns:
        raise ValueError("Expected target column 'quality'.")

    # Features / target
    X = df.drop(columns=["quality"]).to_numpy(dtype=np.float32)
    y = df["quality"].to_numpy(dtype=np.float32)

    n = len(df)
    n_train = int(train_size * n)

    if n_train == 0 or n_train == n:
        raise ValueError(
            f"train_size={train_size} results in {n_train} training samples."
        )

    # --- Split FIRST (same as dataset.take / skip) ---
    X_train, y_train = X[:n_train], y[:n_train]
    X_test, y_test = X[n_train:], y[n_train:]

    # --- Shuffle training only ---
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n_train)
    X_train, y_train = X_train[perm], y_train[perm]

    # --- Convert to torch tensors ---
    X_train_t = torch.from_numpy(X_train)
    y_train_t = torch.from_numpy(y_train)
    X_test_t = torch.from_numpy(X_test)
    y_test_t = torch.from_numpy(y_test)

    train_ds = TensorDataset(X_train_t, y_train_t)
    test_ds = TensorDataset(X_test_t, y_test_t)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=False,  # already shuffled above
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    return train_loader, test_loader


## Compile, train, and evaluate the model

In [3]:
learning_rate = 0.001


def evaluate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    num_samples = 0

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)

            preds = model(X).squeeze()
            loss = loss_fn(preds, y)

            batch_size = X.size(0)
            total_loss += loss.item() * batch_size
            num_samples += batch_size

    mse = total_loss / num_samples
    return np.sqrt(mse)

def run_experiment(model, loss_fn, train_dataloader, test_dataloader, num_epochs):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = RMSprop(model.parameters(), lr=learning_rate)

    print("Start training the model...")

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        num_samples = 0

        for X, y in train_dataloader:
            X = X.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            preds = model(X).squeeze()
            loss = loss_fn(preds, y)
            loss.backward()
            optimizer.step()

            batch_size = X.size(0)
            total_loss += loss.item() * batch_size
            num_samples += batch_size

        # Optional: validation each epoch (like Keras `validation_data`)
        val_rmse = evaluate(model, test_dataloader, loss_fn, device)

    print("Model training finished.")

    train_rmse = evaluate(model, train_dataloader, loss_fn, device)
    print(f"Train RMSE: {round(train_rmse, 3)}")

    print("Evaluating model performance...")
    test_rmse = evaluate(model, test_dataloader, loss_fn, device)
    print(f"Test RMSE: {round(test_rmse, 3)}")


## Create model inputs

In [4]:
FEATURE_NAMES = [
    "fixed acidity",
    "volatile acidity",
    "citric acid",
    "residual sugar",
    "chlorides",
    "free sulfur dioxide",
    "total sulfur dioxide",
    "density",
    "pH",
    "sulphates",
    "alcohol",
]

NUM_FEATURES = len(FEATURE_NAMES)


## Experiment 1: standard neural network

We create a standard deterministic neural network model as a baseline.

In [5]:
hidden_units = [8, 8]
class BaselineModel(nn.Module):
    def __init__(self):
        super().__init__()

        layers = []

        # Equivalent to keras.layers.concatenate + BatchNormalization
        layers.append(nn.BatchNorm1d(NUM_FEATURES))

        input_dim = NUM_FEATURES
        for units in hidden_units:
            layers.append(nn.Linear(input_dim, units))
            layers.append(nn.Sigmoid())
            input_dim = units

        # Output layer: single deterministic point estimate
        layers.append(nn.Linear(input_dim, 1))

        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor):
        # x shape: (batch_size, 11)
        return self.net(x)

Let's split the wine dataset into training and test sets, with 85% and 15% of
the examples, respectively.

In [6]:
train_size = 0.85
batch_size=256
train_dataloader, test_dataloader = get_train_and_test_dataloaders(train_size, batch_size)

Now let's train the baseline model. We use the `MeanSquaredError`
as the loss function.

In [7]:
num_epochs = 100
mse_loss = nn.MSELoss()

baseline_model = BaselineModel()
run_experiment(
    model=baseline_model,
    loss_fn=mse_loss,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    num_epochs=num_epochs,
)

Start training the model...
Model training finished.
Train RMSE: 0.793
Evaluating model performance...
Test RMSE: 0.682


We take a sample from the test set use the model to obtain predictions for them.
Note that since the baseline model is deterministic, we get a single a
*point estimate* prediction for each test example, with no information about the
uncertainty of the model nor the prediction.

In [8]:
sample = 10

baseline_model.eval()

# Take a single batch from the test DataLoader
examples, targets = next(iter(test_dataloader))

device = next(baseline_model.parameters()).device  # cuda or cpu

# Keep only `sample` examples
examples = examples[:sample].to(device)
targets = targets[:sample].to(device)

with torch.no_grad():
    predicted = baseline_model(examples)

for i in range(sample):
    print(
        f"Predicted: {round(predicted[i].item(), 1)} - "
        f"Actual: {targets[i].item()}"
    )


Predicted: 6.0 - Actual: 5.0
Predicted: 6.0 - Actual: 5.0
Predicted: 6.4 - Actual: 7.0
Predicted: 6.3 - Actual: 7.0
Predicted: 6.4 - Actual: 8.0
Predicted: 6.4 - Actual: 6.0
Predicted: 6.4 - Actual: 7.0
Predicted: 5.7 - Actual: 7.0
Predicted: 6.4 - Actual: 5.0
Predicted: 6.2 - Actual: 6.0


## Experiment 2: Bayesian neural network (BNN)

The object of the Bayesian approach for modeling neural networks is to capture
the *epistemic uncertainty*, which is uncertainty about the model fitness,
due to limited training data.

The idea is that, instead of learning specific weight (and bias) *values* in the
neural network, the Bayesian approach learns weight *distributions*
- from which we can sample to produce an output for a given input -
to encode weight uncertainty.

Thus, we need to define prior and the posterior distributions of these weights,
and the training process is to learn the parameters of these distributions.

In [9]:
NUM_FEATURES = 11
hidden_units = [8, 8]

# Prior / posterior init hyperparams (common defaults in bayesian-torch examples)
PRIOR_MU = 0.0
PRIOR_VAR = 1.0
POSTERIOR_MU_INIT = 0.0
POSTERIOR_RHO_INIT = -3.0  # smaller => smaller initial sigma

class BayesianWineMLP(nn.Module):
    def __init__(
        self,
        prior_mean=0.0,
        prior_variance=1.0,
        posterior_mu_init=0.0,
        posterior_rho_init=-3.0,  # smaller => smaller initial sigma
        activation="sigmoid",
    ):
        super().__init__()

        self.bn = nn.BatchNorm1d(NUM_FEATURES)

        self.fc1 = LinearReparameterization(
            in_features=NUM_FEATURES,
            out_features=hidden_units[0],
            prior_mean=prior_mean,
            prior_variance=prior_variance,
            posterior_mu_init=posterior_mu_init,
            posterior_rho_init=posterior_rho_init,
        )
        self.fc2 = LinearReparameterization(
            in_features=hidden_units[0],
            out_features=hidden_units[1],
            prior_mean=prior_mean,
            prior_variance=prior_variance,
            posterior_mu_init=posterior_mu_init,
            posterior_rho_init=posterior_rho_init,
        )
        self.out = LinearReparameterization(
            in_features=hidden_units[1],
            out_features=1,
            prior_mean=prior_mean,
            prior_variance=prior_variance,
            posterior_mu_init=posterior_mu_init,
            posterior_rho_init=posterior_rho_init,
        )

        if activation == "sigmoid":
            self.act = nn.Sigmoid()
        elif activation == "relu":
            self.act = nn.ReLU()
        else:
            raise ValueError("activation must be 'sigmoid' or 'relu'.")

    def forward(self, x):
        kl_sum = 0.0

        x = self.bn(x)

        x, kl = self.fc1(x)
        kl_sum = kl_sum + kl
        x = self.act(x)

        x, kl = self.fc2(x)
        kl_sum = kl_sum + kl
        x = self.act(x)

        x, kl = self.out(x)
        kl_sum = kl_sum + kl

        return x, kl_sum

The epistemic uncertainty can be reduced as we increase the size of the
training data. That is, the more data the BNN model sees, the more it is certain
about its estimates for the weights (distribution parameters).
Let's test this behaviour by training the BNN model on a small subset of
the training set, and then on the full training set, to compare the output variances.

### Train BNN  with a small training subset.

In [10]:
def evaluate_rmse_bnn(model, dataloader, device):
    model.eval()
    se_sum = 0.0
    n = 0

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)

            preds, _ = model(X)
            preds = preds.squeeze(-1)

            se_sum += torch.sum((preds - y) ** 2).item()
            n += y.numel()

    return np.sqrt(se_sum / n)

def train_bnn(
    model,
    train_dataloader,
    test_dataloader,
    num_epochs=100,
    learning_rate=1e-3,
    kl_weight=1.0,
    print_every=10,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate)
    mse_loss = nn.MSELoss()

    # Practical default: scale KL by number of training samples.
    n_train = len(train_dataloader.dataset)
    beta = kl_weight / n_train

    for epoch in range(1, num_epochs + 1):
        model.train()
        epoch_loss = 0.0
        epoch_mse = 0.0
        epoch_kl = 0.0
        n_seen = 0

        for X, y in train_dataloader:
            X = X.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            preds, kl = model(X)
            preds = preds.squeeze(-1)

            data_loss = mse_loss(preds, y)
            loss = data_loss + beta * kl

            loss.backward()
            optimizer.step()

            bs = y.numel()
            epoch_loss += loss.item() * bs
            epoch_mse += data_loss.item() * bs
            epoch_kl += kl.item() * bs
            n_seen += bs

        if (epoch % print_every == 0) or (epoch == 1) or (epoch == num_epochs):
            train_rmse = evaluate_rmse_bnn(model, train_dataloader, device)
            test_rmse = evaluate_rmse_bnn(model, test_dataloader, device)

            print(
                f"Epoch {epoch:3d}/{num_epochs} | "
                f"loss={epoch_loss/n_seen:.4f} | "
                f"mse={epoch_mse/n_seen:.4f} | "
                f"kl={epoch_kl/n_seen:.4f} | "
                f"train_rmse={train_rmse:.3f} | test_rmse={test_rmse:.3f}"
            )

    return model

num_epochs = 500

bnn_model = BayesianWineMLP(activation="sigmoid")  # matches your baseline
bnn_model = train_bnn(
    model=bnn_model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    num_epochs=num_epochs,
    learning_rate=0.001,
    kl_weight=1.0,
    print_every=10,
)


Epoch   1/500 | loss=32.9977 | mse=32.9941 | kl=15.0393 | train_rmse=5.576 | test_rmse=5.576
Epoch  10/500 | loss=14.8832 | mse=14.8795 | kl=15.2557 | train_rmse=3.769 | test_rmse=3.752
Epoch  20/500 | loss=5.6063 | mse=5.6026 | kl=15.5201 | train_rmse=2.311 | test_rmse=2.232
Epoch  30/500 | loss=2.0822 | mse=2.0784 | kl=15.7310 | train_rmse=1.354 | test_rmse=1.361
Epoch  40/500 | loss=0.9347 | mse=0.9309 | kl=15.8867 | train_rmse=0.955 | test_rmse=0.803
Epoch  50/500 | loss=0.8040 | mse=0.8002 | kl=15.8972 | train_rmse=0.899 | test_rmse=0.760
Epoch  60/500 | loss=0.7442 | mse=0.7403 | kl=15.8490 | train_rmse=0.857 | test_rmse=0.756
Epoch  70/500 | loss=0.6722 | mse=0.6684 | kl=15.8305 | train_rmse=0.813 | test_rmse=0.701
Epoch  80/500 | loss=0.6363 | mse=0.6325 | kl=15.9074 | train_rmse=0.804 | test_rmse=0.715
Epoch  90/500 | loss=0.6170 | mse=0.6132 | kl=15.9987 | train_rmse=0.792 | test_rmse=0.690
Epoch 100/500 | loss=0.6120 | mse=0.6082 | kl=16.0922 | train_rmse=0.779 | test_rmse=0

Since we have trained a BNN model, the model produces a different output each time
we call it with the same input, since each time a new set of weights are sampled
from the distributions to construct the network and produce an output.
The less certain the mode weights are, the more variability (wider range) we will
see in the outputs of the same inputs.

In [11]:
def mc_predict(model, X, mc_samples=200):
    device = next(model.parameters()).device
    X = X.to(device)
    model.eval()

    preds = []
    with torch.no_grad():
        for _ in range(mc_samples):
            yhat, _ = model(X)
            preds.append(yhat.squeeze(-1))

    # (mc_samples, batch)
    return torch.stack(preds, dim=0)

Xb, yb = next(iter(test_dataloader))
mc_preds = mc_predict(bnn_model, Xb[:10], mc_samples=200)

for i in range(mc_preds.shape[1]):
    p = mc_preds[:, i]
    mean = p.mean().item()
    min_ = p.min().item()
    max_ = p.max().item()
    range_ = max_ - min_
    actual = yb[i].item()

    print(
        f"Predictions mean: {mean:.2f}, "
        f"min: {min_:.2f}, "
        f"max: {max_:.2f}, "
        f"range: {range_:.2f} - "
        f"Actual: {actual:.1f}"
    )


Predictions mean: 5.32, min: 5.19, max: 5.46, range: 0.28 - Actual: 5.0
Predictions mean: 5.38, min: 5.24, max: 5.52, range: 0.28 - Actual: 5.0
Predictions mean: 6.35, min: 6.18, max: 6.50, range: 0.33 - Actual: 7.0
Predictions mean: 6.54, min: 6.37, max: 6.69, range: 0.32 - Actual: 7.0
Predictions mean: 6.96, min: 6.79, max: 7.09, range: 0.30 - Actual: 8.0
Predictions mean: 6.69, min: 6.50, max: 6.84, range: 0.34 - Actual: 6.0
Predictions mean: 6.79, min: 6.61, max: 6.95, range: 0.33 - Actual: 7.0
Predictions mean: 6.06, min: 5.92, max: 6.16, range: 0.24 - Actual: 7.0
Predictions mean: 6.68, min: 6.50, max: 6.82, range: 0.32 - Actual: 5.0
Predictions mean: 5.95, min: 5.78, max: 6.11, range: 0.33 - Actual: 6.0


## Experiment 3: probabilistic Bayesian neural network

So far, the output of the standard and the Bayesian NN models that we built is
deterministic, that is, produces a point estimate as a prediction for a given example.
We can create a probabilistic NN by letting the model output a distribution.
In this case, the model captures the *aleatoric uncertainty* as well,
which is due to irreducible noise in the data, or to the stochastic nature of the
process generating the data.

In this example, we model the output as a `IndependentNormal` distribution,
with learnable mean and variance parameters. If the task was classification,
we would have used `IndependentBernoulli` with binary classes, and `OneHotCategorical`
with multiple classes, to model distribution of the model output.

In [12]:
NUM_FEATURES = 11
hidden_units = [8, 8]

class ProbabilisticBayesianWineMLP(nn.Module):
    """
    Outputs an Independent Normal distribution:
      y ~ Normal(loc=mu(x), scale=sigma(x))

    Bayesian weights (epistemic) via bayesian-torch layers.
    Aleatoric via learned sigma(x).
    """
    def __init__(
        self,
        prior_mean=0.0,
        prior_variance=1.0,
        posterior_mu_init=0.0,
        posterior_rho_init=-3.0,
        activation="sigmoid",
        min_sigma=1e-3,   # numerical stability floor
    ):
        super().__init__()
        self.min_sigma = min_sigma

        self.bn = nn.BatchNorm1d(NUM_FEATURES)

        self.fc1 = LinearReparameterization(
            in_features=NUM_FEATURES,
            out_features=hidden_units[0],
            prior_mean=prior_mean,
            prior_variance=prior_variance,
            posterior_mu_init=posterior_mu_init,
            posterior_rho_init=posterior_rho_init,
        )
        self.fc2 = LinearReparameterization(
            in_features=hidden_units[0],
            out_features=hidden_units[1],
            prior_mean=prior_mean,
            prior_variance=prior_variance,
            posterior_mu_init=posterior_mu_init,
            posterior_rho_init=posterior_rho_init,
        )

        # Two Bayesian output heads: mean and log-variance (or unconstrained scale param)
        self.mu_head = LinearReparameterization(
            in_features=hidden_units[1],
            out_features=1,
            prior_mean=prior_mean,
            prior_variance=prior_variance,
            posterior_mu_init=posterior_mu_init,
            posterior_rho_init=posterior_rho_init,
        )
        self.rho_head = LinearReparameterization(
            in_features=hidden_units[1],
            out_features=1,
            prior_mean=prior_mean,
            prior_variance=prior_variance,
            posterior_mu_init=posterior_mu_init,
            posterior_rho_init=posterior_rho_init,
        )

        if activation == "sigmoid":
            self.act = nn.Sigmoid()
        elif activation == "relu":
            self.act = nn.ReLU()
        else:
            raise ValueError("activation must be 'sigmoid' or 'relu'.")

    def forward(self, x):
        kl_sum = 0.0

        x = self.bn(x)

        x, kl = self.fc1(x); kl_sum = kl_sum + kl
        x = self.act(x)

        x, kl = self.fc2(x); kl_sum = kl_sum + kl
        x = self.act(x)

        mu, kl = self.mu_head(x); kl_sum = kl_sum + kl
        rho, kl = self.rho_head(x); kl_sum = kl_sum + kl

        # Convert unconstrained rho -> positive sigma (softplus is standard)
        sigma = F.softplus(rho) + self.min_sigma

        # Independent Normal over the last dim (event dim = 1)
        dist = Independent(Normal(loc=mu, scale=sigma), 1)
        return dist, kl_sum

Since the output of the model is a distribution, rather than a point estimate,
we use the [negative loglikelihood](https://en.wikipedia.org/wiki/Likelihood_function)
as our loss function to compute how likely to see the true data (targets) from the
estimated distribution produced by the model.

In [13]:
def train_probabilistic_bnn(
    model,
    train_dataloader,
    test_dataloader,
    num_epochs=100,
    learning_rate=1e-3,
    kl_weight=1.0,
    print_every=10,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate)

    # Practical default KL scaling: beta = kl_weight / N_train
    n_train = len(train_dataloader.dataset)
    beta = kl_weight / n_train

    def eval_rmse_using_mean(dataloader):
        model.eval()
        se_sum = 0.0
        n = 0
        with torch.no_grad():
            for X, y in dataloader:
                X = X.to(device)
                y = y.to(device)
                dist, _ = model(X)
                mu = dist.base_dist.loc.squeeze(-1)  # (batch,)
                se_sum += torch.sum((mu - y) ** 2).item()
                n += y.numel()
        return np.sqrt(se_sum / n)

    print("Start training probabilistic BNN (NLL + beta*KL)...")
    for epoch in range(1, num_epochs + 1):
        model.train()
        nll_sum = 0.0
        kl_sum = 0.0
        total_sum = 0.0
        n_seen = 0

        for X, y in train_dataloader:
            X = X.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            dist, kl = model(X)
            # dist.log_prob expects y shaped like dist event shape; ensure (batch, 1)
            y_event = y.view(-1, 1)

            nll = -dist.log_prob(y_event).mean()     # negative log likelihood
            loss = nll + beta * kl

            loss.backward()
            optimizer.step()

            bs = y.numel()
            nll_sum += nll.item() * bs
            kl_sum += kl.item() * bs
            total_sum += loss.item() * bs
            n_seen += bs

        if (epoch % print_every == 0) or (epoch == 1) or (epoch == num_epochs):
            train_rmse = eval_rmse_using_mean(train_dataloader)
            test_rmse = eval_rmse_using_mean(test_dataloader)
            print(
                f"Epoch {epoch:3d}/{num_epochs} | "
                f"loss={total_sum/n_seen:.4f} | nll={nll_sum/n_seen:.4f} | kl={kl_sum/n_seen:.4f} | "
                f"train_rmse={train_rmse:.3f} | test_rmse={test_rmse:.3f}"
            )

    return model


In [14]:
num_epochs = 500

prob_bnn = ProbabilisticBayesianWineMLP(activation="sigmoid")
prob_bnn = train_probabilistic_bnn(
    model=prob_bnn,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    num_epochs=num_epochs,
    learning_rate=1e-3,
    kl_weight=1.0,
    print_every=10,
)

Start training probabilistic BNN (NLL + beta*KL)...
Epoch   1/500 | loss=17.8863 | nll=17.8815 | kl=20.2156 | train_rmse=5.672 | test_rmse=5.628
Epoch  10/500 | loss=4.3930 | nll=4.3881 | kl=20.4393 | train_rmse=4.514 | test_rmse=4.330
Epoch  20/500 | loss=2.8814 | nll=2.8765 | kl=20.6274 | train_rmse=3.451 | test_rmse=3.334
Epoch  30/500 | loss=2.3496 | nll=2.3446 | kl=20.7644 | train_rmse=2.530 | test_rmse=2.472
Epoch  40/500 | loss=2.0363 | nll=2.0313 | kl=20.8696 | train_rmse=1.602 | test_rmse=1.535
Epoch  50/500 | loss=1.4812 | nll=1.4762 | kl=20.9581 | train_rmse=0.950 | test_rmse=0.807
Epoch  60/500 | loss=1.2590 | nll=1.2540 | kl=20.9523 | train_rmse=0.835 | test_rmse=0.740
Epoch  70/500 | loss=1.2305 | nll=1.2254 | kl=21.0869 | train_rmse=0.807 | test_rmse=0.697
Epoch  80/500 | loss=1.2090 | nll=1.2039 | kl=21.2349 | train_rmse=0.787 | test_rmse=0.713
Epoch  90/500 | loss=1.1940 | nll=1.1889 | kl=21.4071 | train_rmse=0.786 | test_rmse=0.689
Epoch 100/500 | loss=1.1865 | nll=1.

Now let's produce an output from the model given the test examples.
The output is now a distribution, and we can use its mean and variance
to compute the confidence intervals (CI) of the prediction.

In [15]:
def predict_probabilistic_and_print(
    model,
    examples,
    targets,
    mc_samples=200,
):
    device = next(model.parameters()).device
    examples = examples.to(device)
    targets = targets.to(device)

    model.eval()

    mu_samples = []
    sigma_samples = []

    with torch.no_grad():
        for _ in range(mc_samples):
            dist, _ = model(examples)

            # Extract parameters of Independent Normal
            mu = dist.base_dist.loc.squeeze(-1)      # (batch,)
            sigma = dist.base_dist.scale.squeeze(-1) # (batch,)

            mu_samples.append(mu)
            sigma_samples.append(sigma)

    # Stack MC samples
    mu_samples = torch.stack(mu_samples, dim=0)       # (mc, batch)
    sigma_samples = torch.stack(sigma_samples, dim=0) # (mc, batch)

    # Predictive mean
    pred_mean = mu_samples.mean(dim=0)

    # Epistemic uncertainty (from weight sampling)
    epi_std = mu_samples.std(dim=0)

    # Aleatoric uncertainty (expected data noise)
    alea_std = sigma_samples.mean(dim=0)

    # Total predictive uncertainty
    total_std = torch.sqrt(epi_std**2 + alea_std**2)

    # 95% confidence interval
    upper = pred_mean + 1.96 * total_std
    lower = pred_mean - 1.96 * total_std

    # Move to CPU for printing
    pred_mean = pred_mean.cpu()
    total_std = total_std.cpu()
    upper = upper.cpu()
    lower = lower.cpu()
    targets = targets.cpu()

    for idx in range(len(pred_mean)):
        print(
            f"Prediction mean: {pred_mean[idx].item():.2f}, "
            f"stddev: {total_std[idx].item():.2f}, "
            f"95% CI: [{lower[idx].item():.2f} - {upper[idx].item():.2f}] "
            f"- Actual: {targets[idx].item():.1f}"
        )

sample = 10
examples, targets = next(iter(test_dataloader))
examples = examples[:sample]
targets = targets[:sample]

predict_probabilistic_and_print(
    prob_bnn,
    examples,
    targets,
    mc_samples=200,
)

Prediction mean: 5.20, stddev: 0.76, 95% CI: [3.70 - 6.70] - Actual: 5.0
Prediction mean: 5.30, stddev: 0.77, 95% CI: [3.78 - 6.81] - Actual: 5.0
Prediction mean: 6.42, stddev: 0.81, 95% CI: [4.83 - 8.00] - Actual: 7.0
Prediction mean: 6.65, stddev: 0.78, 95% CI: [5.12 - 8.18] - Actual: 7.0
Prediction mean: 6.95, stddev: 0.82, 95% CI: [5.34 - 8.56] - Actual: 8.0
Prediction mean: 6.77, stddev: 0.80, 95% CI: [5.19 - 8.34] - Actual: 6.0
Prediction mean: 6.80, stddev: 0.82, 95% CI: [5.19 - 8.41] - Actual: 7.0
Prediction mean: 6.04, stddev: 0.72, 95% CI: [4.63 - 7.45] - Actual: 7.0
Prediction mean: 6.68, stddev: 0.80, 95% CI: [5.11 - 8.25] - Actual: 5.0
Prediction mean: 5.97, stddev: 0.83, 95% CI: [4.34 - 7.59] - Actual: 6.0
